# Introduction to Django Testing

## Testing Django Applications with pytest
---

In this lesson, you'll learn how to **test Django applications** using pytest.

You know pytest fundamentals - fixtures, parametrization, mocking. Now you'll connect that knowledge with Django.

1. [Why Test Django Applications?](#Why-Test-Django-Applications?),
    - [What Can Go Wrong](#What-Can-Go-Wrong),
    - [Testing Pyramid for Django](#Testing-Pyramid-for-Django),
2. [Setting Up pytest-django](#Setting-Up-pytest-django),
    - [Installation](#Installation),
    - [Configuration in pyproject.toml](#Configuration-in-pyproject.toml),
    - [Optional: pytest.ini](#Optional:-pytest.ini),
3. [Your First Django Test](#Your-First-Django-Test),
    - [Project Structure for Tests](#Project-Structure-for-Tests),
    - [Writing a Simple Test](#Writing-a-Simple-Test),
    - [Running Tests](#Running-Tests),
4. [The Test Database](#The-Test-Database),
    - [How Django Creates Test Database](#How-Django-Creates-Test-Database),
    - [Database Isolation](#Database-Isolation),
    - [Speeding Up Tests](#Speeding-Up-Tests),
5. [Django TestCase vs pytest](#Django-TestCase-vs-pytest),
    - [unittest Style (Django default)](#unittest-Style-(Django-default)),
    - [pytest Style (recommended)](#pytest-Style-(recommended)),
    - [Which to Choose?](#Which-to-Choose?),
6. [Essential conftest.py for Django](#Essential-conftest.py-for-Django),
    - [Basic Django Fixtures](#Basic-Django-Fixtures),
    - [Fixture Scope Considerations](#Fixture-Scope-Considerations),
7. [Practical Example: Testing the blog app](#Practical-Example:-Testing-the-blog-app),
8. [🧠 EXERCISE 🧠, Set Up Testing for a Django Project](#🧠-EXERCISE-🧠,-Set-Up-Testing-for-a-Django-Project),
9. [🧠 EXERCISE 🧠, Write Your First Model Tests](#🧠-EXERCISE-🧠,-Write-Your-First-Model-Tests).

<br>


## Why Test Django Applications?

---

Django applications have many moving parts - models, views, templates, forms, URLs. Each can break independently, and bugs often hide in the **interactions** between components.

<br>

### What Can Go Wrong

---

| Layer | Example Bugs |
|-------|-------------|
| **Models** | Validation doesn't work, `__str__` crashes, custom methods return wrong values |
| **Views** | Wrong template, missing context, incorrect redirect, permission bypass |
| **URLs** | Route doesn't match, wrong view called, parameters not passed |
| **Forms** | Validation missing, wrong fields, data not saved correctly |
| **Templates** | Missing variables, wrong filters, broken layout |

<br>

**Without tests**, you discover these bugs when:
- Users report them (embarrassing)
- Production crashes (expensive)
- You refactor and break things (frustrating)

<br>

### Testing Pyramid for Django

---

```
          /\            E2E Tests (Selenium)
         /  \           - Slow, brittle
        /    \          - Test full user flows
       /------\        
      /        \        Integration Tests (Views, Forms)
     /          \       - Test component interaction
    /            \      - Use test client
   /--------------\
  /                \    Unit Tests (Models, Utils)
 /                  \   - Fast, isolated
/____________________\  - Test single functions
```

<br>

**In this course, we focus on:**
- **Unit tests** for models and utility functions
- **Integration tests** for views and URLs

E2E/browser testing (Selenium/Playwright) is a separate topic.

<br>

## Setting Up pytest-django

---

`pytest-django` is a pytest plugin that provides:
- Integration with Django's test database
- Useful fixtures (`client`, `rf`, `db`)
- Django-specific markers
- Auto-discovery of tests in Django apps

<br>

### Installation

---

**Using uv (recommended):**

```bash
uv add --dev pytest-django
```

`--dev` puts the package in the dev dependency group so a production install (e.g. `uv sync --no-dev`) does not pull in the test stack.

**Using pip:**

```bash
pip install pytest-django
```

(Install into a dev/CI environment; pip has no project-level dev group unless you use extras or separate requirement files.)

<br>

This also installs `pytest` if you don't have it already.

<br>

### Configuration in `pyproject.toml`

---

`pytest-django` needs `DJANGO_SETTINGS_MODULE` (and other pytest options) configured **once**.

In modern projects, put them under **`[tool.pytest.ini_options]`** in the same `pyproject.toml` as your build/deps (next to `manage.py`, or wherever you already keep project metadata)—so you do not maintain a second config file with the same keys.

```bash
# pyproject.toml (snippet)

[tool.pytest.ini_options]
DJANGO_SETTINGS_MODULE = "mysite.settings"
python_files = ["tests.py", "test_*.py", "*_tests.py"]
```

<br>

| Option | Purpose |
|--------|--------|
| `DJANGO_SETTINGS_MODULE` | Import path to your Django settings module |
| `python_files` | Glob patterns for test file discovery |




<br>

### Optional: `pytest.ini` (legacy)

---

Some teams still use a dedicated **`pytest.ini`** next to `manage.py`. Pytest supports it, but **do not copy the same options into both `pytest.ini` and `pyproject.toml`**—you would duplicate configuration and invite drift. Pick **one** file for pytest settings (we recommend `pyproject.toml` in the previous section).

Use `pytest.ini` only when it is clearly the better fit—for example a layout where the Django tree has no `pyproject.toml`, or local conventions require a tiny ini next to `manage.py`.

**Monorepo note:** If the **repository root** `pyproject.toml` sets a narrow `testpaths` (e.g. only top-level scripts), `pytest` may skip `blog/tests/`. Prefer fixing that in **one place**: widen or drop `testpaths` in the root `pyproject.toml`, **or** run `pytest` from the Django project directory whose config is the single source of truth for that app.


<br>

## Your First Django Test

---

### Project Structure for Tests

---

We recommend putting tests **inside each Django app**:

```
mysite/
├── manage.py
├── pyproject.toml            # includes [...ini_options]
├── mysite/
│   ├── settings.py
│   └── urls.py
└── blog/
    ├── models.py
    ├── views.py
    ├── urls.py
    └── tests/                 # Tests directory
        ├── __init__.py
        ├── conftest.py        # App-specific fixtures
        ├── test_models.py     # Model tests
        ├── test_views.py      # View tests
        └── test_urls.py       # URL tests
```

<br>

**Why this structure?**
- Tests live close to the code they test
- Each app is self-contained
- Easy to find relevant tests

<br>

If your Django project is nested inside the Git repository (the directory with `manage.py` is not the repo root), **run `pytest` from that directory** so `DJANGO_SETTINGS_MODULE` and imports such as `from blog.models import Blog` behave the same as `python manage.py`.


<br>

**Create the tests directory:**

**Linux/macOS:**
```bash
mkdir -p blog/tests
touch blog/tests/__init__.py
```

**Windows:**
```powershell
mkdir blog\tests
New-Item blog\tests\__init__.py -ItemType File
```


<br>

### Writing a Simple Test

---

Let's test a model from our **mysite** project. Recall our `Blog` model (see `blog/models.py`):


In [ ]:
# blog/models.py — matches mysite/blog/models.py in the course project

from django.db import models
from django.contrib.auth.models import User


class CategoryType(models.IntegerChoices):
    TECH = 1, 'Technology'
    BUSINESS = 2, 'Business'
    LIFESTYLE = 3, 'Lifestyle'
    NEWS = 4, 'News'


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    author_email = models.EmailField(blank=True)
    slug = models.SlugField(max_length=200, blank=True)
    content = models.TextField(blank=True)
    published_date = models.DateField()

    # IntegerChoices example used in this lesson
    category_type = models.PositiveSmallIntegerField(
        choices=CategoryType.choices,
        default=CategoryType.TECH,
    )

    class Meta:
        ordering = ['-published_date', 'title']
        indexes = [
            models.Index(fields=['slug']),
            models.Index(fields=['category_type', '-published_date']),
        ]

    def __str__(self):
        return self.title


class Comment(models.Model):
    blog = models.ForeignKey(Blog, on_delete=models.CASCADE, related_name='comments')
    content = models.TextField()
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ['-created_at']

    def __str__(self):
        return f"Comment on {self.blog.title}"


class BlogReview(models.Model):
    blog = models.ForeignKey(Blog, on_delete=models.CASCADE, related_name='reviews')
    user = models.ForeignKey(User, on_delete=models.CASCADE, related_name='blog_reviews')
    rating = models.PositiveSmallIntegerField()
    comment = models.TextField(blank=True)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        db_table = 'blog_reviews'
        constraints = [
            models.UniqueConstraint(fields=['blog', 'user'], name='unique_blog_review'),
        ]
        indexes = [
            models.Index(fields=['blog', '-created_at']),
        ]

    def __str__(self):
        return f"Review {self.rating}/5 for {self.blog.title}"


Now write a test for it:

In [ ]:
# blog/tests/test_models.py

import pytest
from datetime import date

from blog.models import Blog


@pytest.mark.django_db
def test_blog_str():
    """Test Blog __str__ returns the title."""
    post = Blog.objects.create(
        title="Django testing tips",
        author="Martin Fowler",
        published_date=date(2026, 3, 15),
    )

    assert str(post) == "Django testing tips"


@pytest.mark.django_db
def test_blog_stores_author():
    """Test author name is stored on the model."""
    post = Blog.objects.create(
        title="Clean code in views",
        author="Robert Martin",
        published_date=date(2026, 4, 1),
    )

    assert post.author == "Robert Martin"


**Key point:** The `@pytest.mark.django_db` marker tells pytest that this test needs database access. Without it, the test will fail with a database access error.

We'll cover this marker in detail in the ORM testing lesson.

<br>

### Running Tests

---

Run tests with pytest:

```bash
# Run all tests
pytest

# Run with verbose output
pytest -v

# Run tests in specific app
pytest blog/

# Run specific test file
pytest blog/tests/test_models.py

# Run specific test function
pytest blog/tests/test_models.py::test_blog_str
```

<br>

**Example output:**

```
$ pytest -v

========================= test session starts =========================
platform darwin -- Python 3.14.0, pytest-9.0.3, pluggy-1.5.0
django: version: 6.0.3, settings: mysite.settings
rootdir: /path/to/mysite
plugins: django-4.11.1
collected 2 items

blog/tests/test_models.py::test_blog_str PASSED            [ 50%]
blog/tests/test_models.py::test_blog_stores_author PASSED      [100%]

========================== 2 passed in 0.45s ==========================
```


<br>

## The Test Database

---

### How Django Creates Test Database

---

When you run Django tests, something important happens. This is how Django works for **any** supported database engine—not only SQLite:

1. Django creates a **separate test database** that is *not* the `NAME` from your normal `DATABASES` entry. *SQLite* usually means another file (e.g. `test_…sqlite3`) or in-memory storage. *Server engines* (PostgreSQL, MySQL, **Microsoft SQL Server**, …) get a **separate database** on the server, often with a `test_` name prefix. On a shared SQL Server, the service account may need rights to create/drop that database.
2. It runs all migrations to set up the schema
3. Tests run against the test database
4. After a successful run, the test database is **destroyed** (unless you keep it on purpose, e.g. `manage.py test --keepdb` or pytest with `--reuse-db`).

<br>

**The database you use for day-to-day work in settings (dev, staging, or production) is not where tests write.**

```
┌────────────────────────┐     ┌────────────────────────┐
│  App DB (`DATABASES`)  │     │  Test database         │
│  file or server DB     │     │  (Django test runner)  │
│                        │     │                        │
│  Your dev/staging/     │     │  Migrations, fixtures, │
│  prod data: not the    │     │  test rows; created &  │
│  target of test writes │     │  dropped after tests*  │
└────────────────────────┘     └────────────────────────┘
```
*Unless you use `keepdb` / `reuse-db` to speed up local runs (see point 4).


<br>

### Database Isolation

---

By default, each test runs in a **database transaction** that gets rolled back:

```
Test 1: BEGIN → Create Blog row → Assert → ROLLBACK
Test 2: BEGIN → Create another Blog row → Assert → ROLLBACK
Test 3: BEGIN → Assert query results → ROLLBACK
```

This means:
- Tests don't affect each other
- No cleanup code needed
- Tests are faster (rollback is cheaper than delete)


<br>

### Speeding Up Tests

---

Database setup can be slow. Here are ways to speed it up:

<br>

**1. Reuse the test database:**

```bash
pytest --reuse-db
```

Keeps the test database between runs. Useful during development.

<br>

**2. Create database once per session:**

```bash
pytest --create-db
```

Forces recreation even with `--reuse-db`.

<br>

**3. Use SQLite for tests** (even if production uses PostgreSQL/SQL Server):

Checking **`'test' in sys.argv`** is a common old snippet for `python manage.py test`, but it is **fragile**: any argument containing the substring `test` can match, and **`pytest` contains `test`**, so it may flip SQLite on for pytest in a way that is easy to misread (works partly by accident).

Prefer one of these:

```python
# settings.py — explicit env flag (good for CI and local)
import os

if os.environ.get("DJANGO_USE_SQLITE_TESTS") == "1":
    DATABASES = {
        "default": {
            "ENGINE": "django.db.backends.sqlite3",
            "NAME": ":memory:",
        }
    }
```

```python
# settings.py — only when pytest is loading Django (typical with pytest-django)
import sys

if "pytest" in sys.modules:
    DATABASES = {
        "default": {
            "ENGINE": "django.db.backends.sqlite3",
            "NAME": ":memory:",
        }
    }
```

```python
# settings.py — Django’s built-in test runner only
import sys

if len(sys.argv) >= 2 and sys.argv[1] == "test":
    DATABASES = {
        "default": {
            "ENGINE": "django.db.backends.sqlite3",
            "NAME": ":memory:",
        }
    }
```

Cleanest long-term: a separate module (e.g. `settings_test.py`) and **`DJANGO_SETTINGS_MODULE`** pointed there only in pytest/CI — no branching inside production `settings.py`.

<br>

## Django `TestCase` vs `pytest`

---

Django has its own test framework based on Python's `unittest`. You'll encounter both styles in the wild.

<br>

### `unittest` Style (Django default)

---

In [ ]:
# Traditional Django testing style

from datetime import date

from django.test import TestCase

from blog.models import Blog


class BlogModelTest(TestCase):
    """Tests for Blog model using Django TestCase."""

    def setUp(self):
        """Set up test data - runs before each test method."""
        self.post = Blog.objects.create(
            title="Django testing tips",
            author="Martin Fowler",
            published_date=date(2026, 3, 15),
        )

    def test_blog_str(self):
        """Test __str__ method."""
        self.assertEqual(str(self.post), "Django testing tips")

    def test_blog_author_field(self):
        """Test author CharField."""
        self.assertEqual(self.post.author, "Martin Fowler")


<br>

### `pytest` Style (recommended)

---

In [ ]:
# Modern pytest style

import pytest
from datetime import date

from blog.models import Blog


@pytest.fixture
def blog_post(db):
    """Create a sample blog post."""
    return Blog.objects.create(
        title="Django testing tips",
        author="Martin Fowler",
        published_date=date(2026, 3, 15),
    )


def test_blog_str(blog_post):
    """Test __str__ method."""
    assert str(blog_post) == "Django testing tips"


def test_blog_author_field(blog_post):
    """Test author name is stored."""
    assert blog_post.author == "Martin Fowler"


**Note:** The `db` fixture is provided by pytest-django. When a fixture depends on `db`, it automatically gets database access.

**Using only db in the factory fixture:** Suitable when almost all data is handled via fixtures (as seen here).

Using `@pytest.mark.django_db` on the test (or test class): Recommended when the test itself executes queries like `Blog.objects.get(...)` without a shared fixture, or when you simply want to be explicit.

<br>

### Which to Choose?

---

| Aspect | Django TestCase | pytest |
|--------|-----------------|--------|
| **Syntax** | Class-based, `self.assertEqual` | Function-based, `assert` |
| **Setup/Teardown** | `setUp()` / `tearDown()` methods | Fixtures |
| **Reusability** | Limited to class inheritance | Fixtures are highly reusable |
| **Parametrization** | Manual loop or subTest | `@pytest.mark.parametrize` |
| **Plugins** | Limited | Rich ecosystem |
| **Output** | Basic | Detailed failure diffs |

<br>

**Our recommendation:** Use **pytest style** for new projects. It's more flexible, more readable, and what you've learned in previous lessons applies directly.

<br>

## Essential `conftest.py` for Django

---

### Basic Django Fixtures

---

Create `conftest.py` in your tests directory or project root for shared fixtures:

In [ ]:
# blog/tests/conftest.py

import pytest
from datetime import date

from blog.models import Blog


@pytest.fixture
def blog_post(db):
    """Create a sample blog post for tests."""
    return Blog.objects.create(
        title="Django testing tips",
        author="Martin Fowler",
        published_date=date(2026, 3, 15),
    )


**Key points:**

1. Each fixture takes `db` as a parameter - this grants database access
2. Fixtures can depend on other fixtures (e.g. a `blog_with_comments` fixture can depend on `blog_post`)
3. pytest-django handles the transaction/rollback automatically

In [ ]:
# mysite/blog/tests/conftest.py
from __future__ import annotations

from datetime import date

import pytest

from blog.models import Blog


@pytest.fixture
def blog_post(db):
    return Blog.objects.create(
        title="Django testing tips",
        author="Martin Fowler",
        published_date=date(2026, 3, 15),
    )

In [ ]:
import pytest
from datetime import date

from blog.models import Blog


def test_blog_str(blog_post):
    assert str(blog_post) == "Django testing tips"


@pytest.mark.django_db
def test_blog_stores_author():
    """Test author name is stored on the model."""
    post = Blog.objects.create(
        title="Clean code in views",
        author="Robert Martin",
        published_date=date(2026, 4, 1),
    )

    assert post.author == "Robert Martin"



<br>

### Fixture Scope Considerations

---

For Django fixtures that create database objects, **be careful with scope**:

```python
# GOOD: function scope (default) - each test gets fresh data
@pytest.fixture
def blog_post(db):
    return Blog.objects.create(...)


# CAREFUL: module scope - data shared across tests in the file
@pytest.fixture(scope="module")
def blog_post(db):
    return Blog.objects.create(...)  # May cause issues!
```

<br>

**Why be careful?**
- With `scope="module"`, the object is created once and reused
- If a test modifies the object, other tests see the modification
- This breaks test isolation

<br>

**Rule of thumb:** Use **function scope** (default) for Django model fixtures unless you have a specific reason not to.


<br>

## Practical Example: Testing the blog app

---


Let's set up a complete testing environment for the **blog** app next to `manage.py`.


**1. Project structure after adding tests:**

```
mysite/
├── manage.py
├── pyproject.toml          # deps + [tool.pytest.ini_options]
├── mysite/
│   ├── __init__.py
│   ├── settings.py
│   ├── urls.py
│   └── wsgi.py
└── blog/
    ├── __init__.py
    ├── admin.py
    ├── models.py
    ├── views.py
    ├── urls.py
    └── tests/
        ├── __init__.py
        ├── conftest.py     # Shared fixtures
        └── test_models.py  # Model tests
```


**2. Pytest configuration (`pyproject.toml` snippet):**

```bash
# pyproject.toml (snippet)

[tool.pytest.ini_options]
DJANGO_SETTINGS_MODULE = "mysite.settings"
python_files = ["tests.py", "test_*.py", "*_tests.py"]
addopts = "-v --tb=short"
```


**3. conftest.py with fixtures:**

In [ ]:
# blog/tests/conftest.py

import pytest
from datetime import date

from blog.models import Blog


@pytest.fixture
def blog_post(db):
    """Sample blog post."""
    return Blog.objects.create(
        title="Django testing tips",
        author="Martin Fowler",
        published_date=date(2026, 3, 15),
    )


@pytest.fixture
def another_blog_post(db):
    """Second post (e.g. for count tests)."""
    return Blog.objects.create(
        title="Another post",
        author="Ada Lovelace",
        published_date=date(2026, 4, 22),
    )


**4. test_models.py:**

In [ ]:
# blog/tests/test_models.py

import pytest
from datetime import date

from blog.models import Blog


class TestBlogModel:
    # --- Previously: one assertion per method (more methods, same ``blog_post`` fixture) ---
    # def test_str(self, blog_post):
    #     assert str(blog_post) == "Django testing tips"
    #
    # def test_published_date(self, blog_post):
    #     assert blog_post.published_date == date(2026, 3, 15)
    # --- Below: same idea (+ ``author``), one function + ``@pytest.mark.parametrize`` ---

    @pytest.mark.parametrize(
        "field,expected",
        [
            pytest.param("str", "Django testing tips", id="str"),
            pytest.param("author", "Martin Fowler", id="author"),
            pytest.param("published_date", date(2026, 3, 15), id="published_date"),
        ],
    )
    def test_blog_post_has_expected_attributes(self, blog_post, field, expected):
        """One test function, several cases — ``@pytest.mark.parametrize``."""
        if field == "str":
            assert str(blog_post) == expected
        else:
            assert getattr(blog_post, field) == expected

    def test_title_max_length_enforced(self, db):
        """Very long titles fail model validation."""
        from django.core.exceptions import ValidationError

        post = Blog(
            title="x" * 201,
            author="Test",
            published_date=date(2026, 1, 1),
        )
        with pytest.raises(ValidationError):
            post.full_clean()

    def test_multiple_posts(self, blog_post, another_blog_post):
        """Two fixtures yield two rows."""
        assert Blog.objects.count() == 2


**5. Run the tests:**

```bash
$ pytest

========================= test session starts =========================
django: version: 6.0.3, settings: mysite.settings_test
collected 5 items

blog/tests/test_models.py::TestBlogModel::test_blog_post_has_expected_attributes[str] PASSED
blog/tests/test_models.py::TestBlogModel::test_blog_post_has_expected_attributes[author] PASSED
blog/tests/test_models.py::TestBlogModel::test_blog_post_has_expected_attributes[published_date] PASSED
blog/tests/test_models.py::TestBlogModel::test_title_max_length_enforced PASSED
blog/tests/test_models.py::TestBlogModel::test_multiple_posts PASSED

========================== 5 passed in 0.52s ==========================
```


<br>

## Summary

---

| Concept | Description |
|---------|-------------|
| **pytest-django** | Plugin that integrates pytest with Django |
| **`[tool.pytest.ini_options]`** | Pytest/Django settings in `pyproject.toml` (single source of truth) |
| **Test database** | Separate database created/destroyed for tests |
| **@pytest.mark.django_db** | Marker for tests that need database access |
| **db fixture** | Built-in fixture that enables database access |
| **conftest.py** | File for shared fixtures |

<br>

**Key principles:**

1. **Use pytest-django** instead of Django's unittest-based testing
2. **Configure `DJANGO_SETTINGS_MODULE`** under `[tool.pytest.ini_options]` in `pyproject.toml` (avoid duplicating the same keys in `pytest.ini`)
3. **Tests are isolated** - each runs in a transaction that's rolled back
4. **Use fixtures** for test data, not setUp methods
5. **Organize tests per app** - `app/tests/test_*.py`

<br>

### 🧠 EXERCISE 🧠, Set Up Testing for a Django Project

---

Set up pytest-django for the **mysite** Django project (the folder that contains `manage.py`).

<br>

**Your task:**

1. Install pytest-django (using uv or pip)
2. Add `[tool.pytest.ini_options]` to `pyproject.toml` (next to `manage.py`) with `DJANGO_SETTINGS_MODULE` and `python_files`—do not maintain a duplicate `pytest.ini` unless you deliberately choose that as your only config file
3. Create the tests directory structure for the `blog` app:
   - `blog/tests/__init__.py`
   - `blog/tests/conftest.py`
   - `blog/tests/test_models.py`
4. Write a simple test that verifies pytest can access the database
5. Run pytest and verify it works


<details>
    <summary>▶️ Solution</summary>



**1. Install pytest-django:**

```bash
# Using uv (dev dependency — not needed in production)
uv add --dev pytest-django

# Or using pip
pip install pytest-django
```

**2. Configure pytest in `pyproject.toml`** (next to `manage.py`; merge into your existing file):

```toml
[tool.pytest.ini_options]
DJANGO_SETTINGS_MODULE = "mysite.settings"
python_files = ["tests.py", "test_*.py", "*_tests.py"]
```

**3. Create tests directory (Linux/macOS):**

```bash
mkdir -p blog/tests
touch blog/tests/__init__.py
touch blog/tests/conftest.py
touch blog/tests/test_models.py
```

**4. Write conftest.py:**

```python
# blog/tests/conftest.py

import pytest
from datetime import date

from blog.models import Blog


@pytest.fixture
def blog_post(db):
    return Blog.objects.create(
        title="Hello from tests",
        author="Test Author",
        published_date=date(2026, 1, 1),
    )
```

**5. Write test_models.py:**

```python
# blog/tests/test_models.py

import pytest
from datetime import date

from blog.models import Blog


@pytest.mark.django_db
def test_database_access():
    """Verify we can access the database."""
    Blog.objects.create(
        title="Smoke test",
        author="CI",
        published_date=date(2026, 1, 1),
    )

    assert Blog.objects.count() == 1


def test_blog_str(blog_post):
    """Test Blog __str__ returns title."""
    assert str(blog_post) == "Hello from tests"
```

**6. Run pytest** (from the directory that contains `manage.py`):

```bash
$ pytest -v

blog/tests/test_models.py::test_database_access PASSED
blog/tests/test_models.py::test_blog_str PASSED
```



</details>


<br>

### 🧠 EXERCISE 🧠, Write Your First Model Tests

---

Write comprehensive tests for the `Blog` model.

<br>

**The models** (see `mysite/blog/models.py` in the repo — the `Blog` excerpt below is the part you will test; `Comment` and `BlogReview` live in the same module):

```python
from django.db import models
from django.contrib.auth.models import User


class CategoryType(models.IntegerChoices):
    TECH = 1, 'Technology'
    BUSINESS = 2, 'Business'
    LIFESTYLE = 3, 'Lifestyle'
    NEWS = 4, 'News'


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    author_email = models.EmailField(blank=True)
    slug = models.SlugField(max_length=200, blank=True)
    content = models.TextField(blank=True)
    published_date = models.DateField()

    category_type = models.PositiveSmallIntegerField(
        choices=CategoryType.choices,
        default=CategoryType.TECH,
    )

    class Meta:
        ordering = ['-published_date', 'title']
        indexes = [
            models.Index(fields=['slug']),
            models.Index(fields=['category_type', '-published_date']),
        ]

    def __str__(self):
        return self.title
```

Optional fields (`author_email`, `slug`, `content`) and `category_type` (default `TECH`) do not need to be passed to `Blog.objects.create()` in basic tests.

<br>

**Your task:**

1. Create a `blog_post` fixture in `conftest.py`
2. Write tests for:
   - `__str__` returns the title
   - `author` and `published_date` are stored correctly
   - `title` longer than 200 characters fails `full_clean()` with `ValidationError`
   - Multiple posts can be created, and the next test still sees an empty DB (isolation)
3. Use a test class to organize the tests


<details>
    <summary>▶️ Solution</summary>


```python
# blog/tests/conftest.py

import pytest
from datetime import date

from blog.models import Blog


@pytest.fixture
def blog_post(db):
    """Create a sample blog post."""
    return Blog.objects.create(
        title="Django testing tips",
        author="Martin Fowler",
        published_date=date(2026, 3, 15),
    )
```

---

```python
# blog/tests/test_models.py

import pytest
from datetime import date

from django.core.exceptions import ValidationError

from blog.models import Blog


class TestBlogModel:
    """Tests for Blog model."""

    # Previously (separate methods):
    # def test_str_returns_title(self, blog_post):
    #     assert str(blog_post) == "Django testing tips"
    #
    # def test_stores_author_and_date(self, blog_post):
    #     assert blog_post.author == "Martin Fowler"
    #     assert blog_post.published_date == date(2026, 3, 15)

    @pytest.mark.parametrize(
        "field,expected",
        [
            pytest.param("str", "Django testing tips", id="str"),
            pytest.param("author", "Martin Fowler", id="author"),
            pytest.param("published_date", date(2026, 3, 15), id="published_date"),
        ],
    )
    def test_blog_post_has_expected_attributes(self, blog_post, field, expected):
        if field == "str":
            assert str(blog_post) == expected
        else:
            assert getattr(blog_post, field) == expected

    def test_title_max_length(self, db):
        post = Blog(
            title="x" * 201,
            author="Test",
            published_date=date(2026, 1, 1),
        )
        with pytest.raises(ValidationError):
            post.full_clean()

    def test_multiple_posts(self, db):
        Blog.objects.create(title="A", author="a", published_date=date(2026, 1, 1))
        Blog.objects.create(title="B", author="b", published_date=date(2026, 1, 2))
        assert Blog.objects.count() == 2

    def test_each_test_starts_with_empty_db(self, db):
        """pytest-django rolls back DB between tests — no leftover rows."""
        assert Blog.objects.count() == 0
```



</details>


---